# 6.8 訓練窗口是在切詞嗎？


已有ID0到9，模型每題只看三格。取[0,1,2,3]後，X=[0,1,2]、Y=[1,2,3]；切段決定本題前文，沒有改ID含義。

這種段叫訓練窗口。C格输入和C格右移答案需要原C+1格，再shift一次；切詞已在此前完成。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from tiny_perceptron.data import shifted

ids = list(range(10))
context = 3
for start in [0, 3, 6]:
    segment = ids[start : start + context + 1]
    x, y = shifted(segment)
    print("起點", start, "輸入", x.tolist(), "答案", y.tolist())
    assert len(x) == len(y) == context

輸入十個已有ID與窗口長度3，`range(10)` 建0到9的一排整數；本例它們只當容易追蹤的代號。三起點分別取0–3、3–6、6–9，共各四格；shift後輸出 `(0,1,2)→(1,2,3)`、`(3,4,5)→(4,5,6)`、`(6,7,8)→(7,8,9)`。`.tolist()` 將tensor轉回普通清單方便顯示，檢查每題確實三輸入、三答案。

相鄰窗口共用邊界token3與6，但它們提供的位置角色不同；本例所有題屬於同一份已分組訓練資料，允許這種共用。不能先從同一文件取這些高度重疊窗口，再隨機分訓練與驗證，[1.2](https://birdhackor.github.io/tiny-perceptron-vlm/1.2.html) 已說明原因。也不能跨文件邊界直接連成一個「上一文件末尾預測下一文件開頭」目標，除非明確設計並標記這種邊界。

先按完整文件／家族分側，再在各侧取窗口，避免重疊短段跨訓練與驗證。末尾不足C+1時題數會少；丟尾段或補齊都要明說。固定原文byte和固定token窗口也不同，後者可能讓BPE讀更多原文，改變資料預算。

練習只把context改成4，保留三個起點，先預測起點6只有 `[6,7,8,9]` 四個原ID，shift後只有三道題，最後檢查失敗。執行核對，再把起點列表改成 `[0,4]`，兩次都可取五格、建立四題。這次沒有改tokenizer，只改每題資料取段的契約。

<details>
<summary>補充：實作約定與原始紀錄</summary>

這次[6.5的兩工具比較](https://birdhackor.github.io/tiny-perceptron-vlm/6.5.html)還把「取原文多少」與「模型可看幾格」分開：先把完整篇章分側，再取同一份最多256byte的片段，兩工具各自編碼，模型最多容納384格，因此每篇片段都能放進同一題，沒有因BPE較短就多取後面的文字。這讓兩模型的原文曝光相同；若只固定256個token，BPE那側通常會收到更多原文，就已經換了資料預算。

短程式沿用本課工具與原計算語義。安裝、長訓練與重做操作見[訓練配方](https://birdhackor.github.io/tiny-perceptron-vlm/training.html)，不需要先完成長配方才能閱讀這個例子。

</details>
